# Compare-run 분석 — Wilcoxon + 카테고리별 gain + relaxed verdict

`scripts/run_compare.py` 이 저장한 `<output>/summary.csv` 를 읽어 alignment 이 age-trend 를 얼마나 강화했는지 정량.

**핵심 지표**
- `time_r_naive`, `time_r_aligned` — feature 별 (patient, eye) 평균 |Pearson r(time, value)|
- `time_r_gain = aligned - naive` — alignment 이 age trend 를 끌어올린 정도

**분석 3 단계**
1. Paired Wilcoxon (H1: aligned > naive)
2. 카테고리별 mean gain + % positive — whole vs sholl/faz/zone 대비
3. Strict vs Relaxed verdict 비교

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

# ── compare_run 위치 ──
COMPARE_RUN = Path('g:/RAG/project_1/output/compare_run')
SUMMARY_CSV = COMPARE_RUN / 'summary.csv'
assert SUMMARY_CSV.exists(), f'{SUMMARY_CSV} 없음 - run_compare.py 먼저 실행'

df = pd.read_csv(SUMMARY_CSV)
print(f'summary.csv rows: {len(df)}, cols: {list(df.columns)}')
df.head()

## 1. Feature category 태깅

prefix 로 6 그룹 (sholl / faz+cross / zone / knudtson / whole / other).

In [ ]:
ZONE_PREFIXES = ('S1_','N1_','I1_','T1_','S2_','N2_','I2_','T2_','C_','zone_B_','zone_C_')

def categorize(c: str) -> str:
    if c.startswith('sholl'):                       return 'sholl'
    if 'FAZ' in c or c.startswith('crossings'):     return 'faz/cross'
    if c.startswith(ZONE_PREFIXES):                 return 'zone'
    if c.startswith(('CRAE','CRVE','AVR')):         return 'knudtson'
    if c.startswith('whole_'):                      return 'whole'
    return 'other'

df['category'] = df['feature'].map(categorize)
print(df['category'].value_counts())

## 2. Paired Wilcoxon — aligned time_r > naive time_r?

In [ ]:
tr = df.dropna(subset=['time_r_aligned', 'time_r_naive']).copy()
diff = tr['time_r_aligned'] - tr['time_r_naive']

print('=' * 55)
print('Paired time_r_gain (aligned - naive)')
print('=' * 55)
print(f'  N features                  : {len(diff)}')
print(f'  mean gain                   : {diff.mean():+.4f}')
print(f'  median gain                 : {diff.median():+.4f}')
print(f'  # positive (aligned > naive): {(diff > 0).sum()} ({(diff > 0).mean()*100:.1f}%)')
print(f'  # gain > 0.05               : {(diff > 0.05).sum()}')
print(f'  # gain > 0.10               : {(diff > 0.10).sum()}')

w = stats.wilcoxon(tr['time_r_aligned'], tr['time_r_naive'], alternative='greater')
sig = ' ← 유의 (p<0.05)' if w.pvalue < 0.05 else ''
print(f'\n  Wilcoxon signed-rank (H1: aligned > naive)')
print(f'    statistic = {w.statistic:.1f}')
print(f'    p-value   = {w.pvalue:.2e}{sig}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(diff, bins=40, color='#4caf50', alpha=0.7, edgecolor='black')
ax.axvline(0, color='k', ls='--', label='gain = 0')
ax.axvline(diff.mean(), color='red', ls='--', label=f'mean = {diff.mean():+.3f}')
ax.set_xlabel('time_r_gain (aligned - naive)')
ax.set_ylabel('# features')
ax.set_title(f'Distribution of time_r_gain  (N={len(diff)}, Wilcoxon p={w.pvalue:.1e})')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3. 카테고리별 gain bar chart

**예측**: whole-image aggregate 는 alignment 무관 (rotation-invariant), spatial (sholl/faz/zone) 은 이득.

In [ ]:
cat_stats = (tr.groupby('category')
                .agg(n=('time_r_gain','count'),
                     mean_gain=('time_r_gain','mean'),
                     median_gain=('time_r_gain','median'),
                     pct_positive=('time_r_gain', lambda x: (x > 0).mean() * 100))
                .sort_values('mean_gain', ascending=False))
print('카테고리별 gain 분포:')
print(cat_stats.round(3))

colors_cat = ['#2e7d32' if g > 0.02 else '#ffb300' if g > 0 else '#d32f2f'
              for g in cat_stats['mean_gain']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(cat_stats.index, cat_stats['mean_gain'], color=colors_cat, edgecolor='black')
axes[0].axvline(0, color='k', lw=0.8)
axes[0].set_xlabel('mean time_r_gain (aligned - naive)')
axes[0].set_title('카테고리별 평균 gain\n(양수 = alignment 이 age trend 강화)')
for i, (idx, row) in enumerate(cat_stats.iterrows()):
    offset = 0.001 if row['mean_gain'] >= 0 else -0.001
    ha = 'left' if row['mean_gain'] >= 0 else 'right'
    axes[0].text(row['mean_gain'] + offset, i, f" n={int(row['n'])}",
                 va='center', ha=ha, fontsize=9)
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(cat_stats.index, cat_stats['pct_positive'], color=colors_cat, edgecolor='black')
axes[1].axvline(50, color='k', ls='--', lw=0.8, label='50% (chance)')
axes[1].set_xlabel('% features with aligned > naive')
axes[1].set_title('카테고리별 positive gain 비율')
axes[1].legend(); axes[1].grid(axis='x', alpha=0.3)
axes[1].set_xlim(0, 100)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplot per category
cats_sorted = cat_stats.index.tolist()
fig, ax = plt.subplots(figsize=(11, 5))
data = [tr[tr['category'] == c]['time_r_gain'].dropna().values for c in cats_sorted]
bp = ax.boxplot(data, labels=cats_sorted, patch_artist=True, showmeans=True,
                 meanprops=dict(marker='D', markerfacecolor='red', markeredgecolor='red'))
for patch, c in zip(bp['boxes'], colors_cat):
    patch.set_facecolor(c); patch.set_alpha(0.6)
ax.axhline(0, color='k', ls='--', alpha=0.5)
ax.set_ylabel('time_r_gain (aligned - naive)')
ax.set_title('카테고리별 time_r_gain 분포 (box + red diamond = mean)')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

## 4. Relaxed verdict — strict 대비 완화 임계값

**Strict** (run_compare.py 기본): `aligned ≥0.5 AND naive <0.3` → 30명 표본에선 revealed = 0
**Relaxed**: `aligned ≥0.4 AND gain ≥0.10` → alignment 이 age trend 를 유의하게 끌어올린 feature 검출

In [ ]:
def strict_verdict(row):
    ra, rn = row['time_r_aligned'], row['time_r_naive']
    if pd.isna(ra) or pd.isna(rn): return 'unknown'
    if ra >= 0.5 and rn <  0.3: return 'revealed'
    if ra >= 0.5 and rn >= 0.5: return 'both_trend'
    if ra <  0.3 and rn <  0.3: return 'stationary'
    if ra <  0.3 and rn >= 0.5: return 'erased'
    return 'mixed'

def relaxed_verdict(row):
    ra, rn = row['time_r_aligned'], row['time_r_naive']
    if pd.isna(ra) or pd.isna(rn): return 'unknown'
    gain = ra - rn
    if ra >= 0.4 and gain >= 0.10: return 'revealed'
    if ra >= 0.4 and rn >= 0.4:    return 'both_trend'
    if ra <  0.3 and rn <  0.3:    return 'stationary'
    if rn >= 0.4 and ra <  0.3:    return 'erased'
    return 'mixed'

tr['strict_verdict']  = tr.apply(strict_verdict, axis=1)
tr['relaxed_verdict'] = tr.apply(relaxed_verdict, axis=1)

order = ['revealed','both_trend','stationary','erased','mixed','unknown']
cmp = pd.DataFrame({
    'strict':  tr['strict_verdict'].value_counts(),
    'relaxed': tr['relaxed_verdict'].value_counts(),
}).reindex(order).fillna(0).astype(int)
print(cmp)

In [ ]:
# Side-by-side pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
emoji = {'revealed':'⭐','both_trend':'➖','stationary':'📉','erased':'⚠️','mixed':'🔵','unknown':'?'}
colors_map = {'revealed':'#4caf50','both_trend':'#9e9e9e','stationary':'#03a9f4',
              'erased':'#f44336','mixed':'#ba68c8','unknown':'#dddddd'}
for ax, col in zip(axes, ['strict', 'relaxed']):
    d = cmp[col][cmp[col] > 0]
    ax.pie(d.values, labels=[f'{emoji[k]} {k} ({v})' for k, v in d.items()],
           colors=[colors_map[k] for k in d.index], startangle=90,
           autopct=lambda p: f'{p:.0f}%' if p >= 3 else '')
    ax.set_title(f'{col.upper()} verdict  (N={d.sum()})')
plt.tight_layout(); plt.show()

## 5. Relaxed revealed feature 목록 + 카테고리 비율

In [ ]:
rev = tr[tr['relaxed_verdict'] == 'revealed'].sort_values('time_r_gain', ascending=False)
print(f'⭐ Relaxed revealed: {len(rev)} features\n')
if len(rev):
    display(rev[['feature','category','time_r_naive','time_r_aligned',
                  'time_r_gain']].round(3).reset_index(drop=True))

    # 카테고리별 revealed 비율
    fig, ax = plt.subplots(figsize=(9, 4))
    rev_by_cat = rev['category'].value_counts().reindex(cats_sorted, fill_value=0)
    total_by_cat = tr['category'].value_counts().reindex(cats_sorted, fill_value=0)
    pct = (rev_by_cat / total_by_cat.replace(0, np.nan) * 100).fillna(0)
    ax.bar(cats_sorted, pct, color=colors_cat, edgecolor='black')
    for i, c in enumerate(cats_sorted):
        ax.text(i, pct[c] + 1, f"{int(rev_by_cat[c])}/{int(total_by_cat[c])}",
                ha='center', fontsize=9)
    ax.set_ylabel("% features 'revealed' (relaxed)")
    ax.set_title("카테고리별 relaxed revealed 비율")
    plt.xticks(rotation=30, ha='right'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.show()

# 저장
out_csv = COMPARE_RUN / 'trend_revelation_relaxed.csv'
tr.to_csv(out_csv, index=False)
print(f'\n[저장] {out_csv}')